In [0]:
import requests

In [0]:
def read_from_api(api_url, headers):
    response = requests.get(api_url, headers=headers)
    data = response.json()
    df = pd.DataFrame(data)
    return df

api_url = "https://api.example.com/data"
api_headers = {"Authorization": "Bearer <TOKEN>"}
df_api = read_from_api(api_url, api_headers)

In [0]:
{
    "scope": "dbr-key-vault-scope",
    "spn_client_id": "dbr-spn-application-id",
    "spn_secret": "dbr-spn-secret",
    "tenant_id": "tenant-id",
    "expriy_of_token": 100000,
    "comment": "Bearer Token For Poc",
    "key_vault_url": "https://data-platform-dv-uks-01.vault.azure.net/",
    "databricks_rul": "https://adb-12320343928e13355.15.azuredatabricks.net",
    "bearer_token_name": "dbr-bearer-token-for-Poc"
}  

In [0]:
import requests

tenant_id =  dbutils.secrets.get(scope = scope, key = tenant_id  )

url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
payload = {
    'client_id': dbutils.secrets.get(scope = scope, key = spn_client_id  ), 
    'grant_type': 'client_credentials',
    'scope': '2ff814a6-3304-4ab8-85cb-cd0e6f879c1d/.default', # do not change
    'client_secret': dbutils.secrets.get(scope = scope, key = spn_secret  )
}
response = requests.post(url, data=payload)
print(response)
access_token = response.json()["access_token"]

In [0]:
import json

url = "{databricks_url}/api/2.0/token/create"

payload = json.dumps({
  "lifetime_seconds": expriy_of_token,
  "comment": comment
})
headers = {
  'Authorization': f'Bearer {access_token}',
  'Content-Type': 'application/json'
}
response = requests.request("POST", url, headers=headers, data=payload)
dbr_token = response.json()["token_value"]
5. Upload/Set Databricks Bearer Token in to the Azure Key Vault

In [0]:
from azure.identity import ClientSecretCredential
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = ClientSecretCredential(
    tenant_id = dbutils.secrets.get("dbr-key-vault-scope","tenant-id"),
    client_id = dbutils.secrets.get("dbr-key-vault-scope","dbr-spn-application-id"),
    client_secret = dbutils.secrets.get("dbr-key-vault-scope","dbr-spn-secret")
)

credential = DefaultAzureCredential()

secret_client = SecretClient(key_vault_url , credential=credential)
secret = secret_client.set_secret(bearer_token_name, dbr_token)

print(f"secret_name: {secret.name}")
print(f"secret_version: {secret.properties.version}")
print(f"secret_id: {secret.id}")

**Data Extraction from API**

In [0]:
from pyspark.sql import SparkSession
import requests
import json

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Read API Data") \
    .getOrCreate()

def read_from_api(api_url, headers):
    # Make the API request
    response = requests.get(api_url, headers=headers)
    data = response.json()  # Parse the JSON response
    
    # Convert JSON data to an RDD and then to a Spark DataFrame
    rdd = spark.sparkContext.parallelize([data])  # Parallelize the JSON data
    df = spark.read.json(rdd)  # Create a Spark DataFrame from JSON RDD
    return df

# Read from API
api_url = "https://api.example.com/data"
api_headers = {"Authorization": "Bearer <TOKEN>"}
df_api = read_from_api(api_url, api_headers)

# Show the DataFrame
df_api.show()

In [0]:
from pyspark.sql import SparkSession
import requests

spark = SparkSession.builder.appName("API Pagination").getOrCreate()

def fetch_paginated_data(api_url, headers, limit=50):
    offset = 0
    all_data = []

    while True:
        paginated_url = f"{api_url}?offset={offset}&limit={limit}"
        response = requests.get(paginated_url, headers=headers)
        response.raise_for_status()
        data = response.json()
        all_data.extend(data)
        if len(data) < limit:
            break
        offset += limit

    # Load data into PySpark DataFrame
    rdd = spark.sparkContext.parallelize(all_data)
    df = spark.read.json(rdd)
    return df

api_url = "https://api.example.com/data"
headers = {"Authorization": "Bearer <TOKEN>"}
spark_df = fetch_paginated_data(api_url, headers)
spark_df.show()
